In [ ]:
# Compares open-vocabulary object detectors on the manually annotated benchmark.

# Module 3 — comparaison scientifique des détecteurs d'objets

Ce notebook compare **Grounding DINO**, **YOLO-World** et **OWLv2** sur le
benchmark annoté manuellement dans CVAT.

## Protocole expérimental

- 96 images provenant de 24 vidéos, soit 4 images par vidéo ;
- 48 images de **calibration** et 48 images de **test**, séparées par vidéo ;
- mêmes catégories textuelles et mêmes images pour les trois modèles ;
- la classe `child` est exclue de cette comparaison : elle reste disponible
  pour l'évaluation ultérieure de la proximité enfant–objet ;
- association prédiction/vérité terrain par classe, avec
  $\mathrm{IoU} \geq 0{,}50$ et appariement un-à-un ;
- suppression non maximale (NMS) commune, par classe, avec un seuil IoU de 0,50 ;
- seuil de confiance choisi **séparément pour chaque modèle** sur le lot de
  calibration, en maximisant le F1 ; en cas d'égalité, le rappel est privilégié ;
- évaluation finale unique sur le lot de test, sans réajuster les seuils ;
- latence mesurée avec un lot de taille 1, après préchauffage, en excluant
  le chargement des modèles et la lecture des fichiers sur disque.

Les métriques finales sont : précision, rappel, F1 micro, F1 macro,
mAP@0.50, faux positifs par image, images sans détection, latence moyenne,
médiane et 95e percentile.

**Préparation Kaggle :** activer Internet et un GPU, puis ajouter comme
entrées le dataset `Training`, `benchmark_frames.csv` et
`Detector_Benchmark_96_COCO.zip`.

In [ ]:
# CELLULE 1 — Installation
# Ne pas réinstaller PyTorch : la version GPU de Kaggle est conservée.
%pip install -q "transformers>=4.57,<5" "ultralytics>=8.3" pycocotools accelerate

In [ ]:
# CELLULE 2 — Imports, reproductibilité et configuration
import gc
import json
import math
import os
import random
import re
import shutil
import time
import warnings
import zipfile
from collections import defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("/kaggle/working/detector_benchmark_comparison")
FRAME_DIR = OUTPUT_DIR / "frames"
PREDICTION_DIR = OUTPUT_DIR / "predictions"
FIGURE_DIR = OUTPUT_DIR / "figures"
for directory in [OUTPUT_DIR, FRAME_DIR, PREDICTION_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RAW_CONFIDENCE = 0.01
NMS_IOU = 0.50
MATCH_IOU = 0.50
THRESHOLD_GRID = np.round(np.arange(0.01, 1.00, 0.01), 2)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
print("Sorties :", OUTPUT_DIR)

In [ ]:
# ============================================================
# CELLULE 3 — LOCALISATION DES DONNÉES
# ET RECONSTRUCTION DES 96 IMAGES
# ============================================================

from pathlib import Path
import json
import cv2
import pandas as pd

INPUT_ROOT = Path("/kaggle/input")


# ------------------------------------------------------------
# 1. Recherche d'un fichier dans les Inputs Kaggle
# ------------------------------------------------------------

def find_unique(filename):
    matches = sorted(INPUT_ROOT.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"{filename} est introuvable dans /kaggle/input. "
            "Vérifie que le dataset correspondant est ajouté au notebook."
        )

    if len(matches) > 1:
        print(
            f"ATTENTION : Plusieurs fichiers nommés {filename} ont été trouvés. "
            f"Utilisation de : {matches[0]}"
        )

    return matches[0]


# ------------------------------------------------------------
# 2. Localisation du manifeste CSV
# ------------------------------------------------------------

MANIFEST_PATH = find_unique("benchmark_frames.csv")
manifest = pd.read_csv(MANIFEST_PATH)

required_columns = {
    "frame_id",
    "video_id",
    "class_name",
    "benchmark_split",
    "video_filename",
    "video_path",
    "original_frame_index",
    "frame_filename",
    "width",
    "height",
}

missing_columns = required_columns - set(manifest.columns)

assert not missing_columns, (
    f"Colonnes absentes du manifeste : {missing_columns}"
)

assert len(manifest) == 96, (
    f"96 images étaient attendues, mais {len(manifest)} lignes "
    "ont été trouvées dans le manifeste."
)

assert manifest["frame_id"].is_unique, (
    "Les identifiants frame_id ne sont pas uniques."
)

assert manifest["frame_filename"].is_unique, (
    "Les noms des images ne sont pas uniques."
)

assert set(manifest["benchmark_split"]) == {"calibration", "test"}, (
    "Les ensembles calibration et test n'ont pas été trouvés correctement."
)


# ------------------------------------------------------------
# 3. Localisation des annotations COCO
# ------------------------------------------------------------
# Kaggle a automatiquement décompressé l'archive ZIP.
# Nous recherchons donc directement instances_default.json.

coco_candidates = sorted(
    INPUT_ROOT.rglob("instances_default.json")
)

benchmark_candidates = [
    path
    for path in coco_candidates
    if "detector_benchmark_96_coco" in str(path).lower()
]

if benchmark_candidates:

    if len(benchmark_candidates) > 1:
        print(
            "ATTENTION : Plusieurs annotations du benchmark ont été trouvées :"
        )

        for path in benchmark_candidates:
            print(" -", path)

        print(
            "Utilisation de la première :",
            benchmark_candidates[0],
        )

    COCO_PATH = benchmark_candidates[0]

elif len(coco_candidates) == 1:

    COCO_PATH = coco_candidates[0]

else:

    raise FileNotFoundError(
        "Le fichier instances_default.json est introuvable "
        "ou plusieurs fichiers ambigus ont été trouvés.\n"
        f"Candidats détectés : {coco_candidates}"
    )


with open(COCO_PATH, "r", encoding="utf-8") as handle:
    coco = json.load(handle)


# ------------------------------------------------------------
# 4. Résolution des chemins des vidéos originales
# ------------------------------------------------------------

def resolve_video(row):
    """
    Retrouve la vidéo originale dans le dataset historique Training.

    Training2 est volontairement exclu puisque le pipeline final utilise
    le modèle et les données historiques Training.
    """

    # Première possibilité : le chemin contenu dans le manifeste existe.
    exact_path = Path(str(row.video_path))

    if exact_path.exists():
        return exact_path

    # Deuxième possibilité : chemin normalement utilisé dans Kaggle.
    preferred_path = (
        INPUT_ROOT
        / "datasets"
        / "nisrineamalkhnifass"
        / "training"
        / "Training"
        / str(row.class_name)
        / str(row.video_filename)
    )

    if preferred_path.exists():
        return preferred_path

    # Recherche générale en excluant explicitement Training2.
    candidates = [
        path
        for path in INPUT_ROOT.rglob(str(row.video_filename))
        if path.parent.name == str(row.class_name)
        and "training2" not in str(path).lower()
    ]

    if len(candidates) == 0:
        raise FileNotFoundError(
            "Vidéo introuvable : "
            f"{row.class_name}/{row.video_filename}"
        )

    if len(candidates) > 1:
        raise FileNotFoundError(
            "Plusieurs vidéos correspondent au même nom : "
            f"{row.class_name}/{row.video_filename}\n"
            f"Candidats : {candidates}"
        )

    return candidates[0]


resolved_videos = {}

unique_videos = manifest.drop_duplicates("video_id")

for row in unique_videos.itertuples(index=False):
    resolved_videos[row.video_id] = resolve_video(row)


# ------------------------------------------------------------
# 5. Extraction des 96 frames originales
# ------------------------------------------------------------

extraction_errors = []

for row in manifest.itertuples(index=False):

    destination = FRAME_DIR / row.frame_filename

    # Si l'image existe déjà et possède les bonnes dimensions,
    # il n'est pas nécessaire de l'extraire une seconde fois.
    if destination.exists():

        existing_image = cv2.imread(str(destination))

        if (
            existing_image is not None
            and existing_image.shape[1] == int(row.width)
            and existing_image.shape[0] == int(row.height)
        ):
            continue

    video_path = resolved_videos[row.video_id]

    capture = cv2.VideoCapture(str(video_path))

    capture.set(
        cv2.CAP_PROP_POS_FRAMES,
        int(row.original_frame_index),
    )

    success, frame = capture.read()
    capture.release()

    if not success:

        extraction_errors.append(
            {
                "frame_id": row.frame_id,
                "problem": "lecture impossible",
                "video_path": str(video_path),
            }
        )

        continue

    obtained_height, obtained_width = frame.shape[:2]

    if (
        obtained_width != int(row.width)
        or obtained_height != int(row.height)
    ):

        extraction_errors.append(
            {
                "frame_id": row.frame_id,
                "problem": (
                    f"dimensions obtenues : "
                    f"{obtained_width}x{obtained_height} ; "
                    f"dimensions attendues : "
                    f"{row.width}x{row.height}"
                ),
                "video_path": str(video_path),
            }
        )

        continue

    write_success = cv2.imwrite(
        str(destination),
        frame,
    )

    if not write_success:

        extraction_errors.append(
            {
                "frame_id": row.frame_id,
                "problem": "écriture de l'image impossible",
                "destination": str(destination),
            }
        )


assert not extraction_errors, (
    "Certaines images n'ont pas pu être extraites :\n"
    f"{extraction_errors[:10]}"
)


# ------------------------------------------------------------
# 6. Vérifications du benchmark reconstruit
# ------------------------------------------------------------

extracted_images = [
    path
    for path in FRAME_DIR.iterdir()
    if path.is_file()
    and path.suffix.lower() in {
        ".jpg",
        ".jpeg",
        ".png",
        ".bmp",
        ".webp",
    }
]

assert len(extracted_images) == 96, (
    f"96 images étaient attendues dans {FRAME_DIR}, "
    f"mais {len(extracted_images)} ont été trouvées."
)

image_by_id = {
    image["id"]: image
    for image in coco["images"]
}

coco_filenames = {
    image["file_name"]
    for image in coco["images"]
}

manifest_filenames = set(
    manifest["frame_filename"]
)

assert len(coco["images"]) == 96, (
    f"Le fichier COCO contient {len(coco['images'])} images "
    "au lieu des 96 attendues."
)

assert coco_filenames == manifest_filenames, (
    "Les noms des images du manifeste ne correspondent pas "
    "aux noms contenus dans les annotations COCO.\n"
    f"Absents du COCO : {manifest_filenames - coco_filenames}\n"
    f"Absents du manifeste : {coco_filenames - manifest_filenames}"
)

calibration_count = int(
    (manifest["benchmark_split"] == "calibration").sum()
)

test_count = int(
    (manifest["benchmark_split"] == "test").sum()
)

assert calibration_count == 48, (
    f"48 images de calibration attendues, "
    f"{calibration_count} trouvées."
)

assert test_count == 48, (
    f"48 images de test attendues, {test_count} trouvées."
)


# ------------------------------------------------------------
# 7. Résumé
# ------------------------------------------------------------

print()
print("=" * 70)
print("PRÉPARATION DU BENCHMARK TERMINÉE")
print("=" * 70)

print("Manifeste        :", MANIFEST_PATH)
print("Annotations COCO :", COCO_PATH)
print("Dossier images   :", FRAME_DIR)

print()
print("Nombre de vidéos :", manifest["video_id"].nunique())
print("Nombre d'images  :", len(manifest))
print("Annotations      :", len(coco["annotations"]))

print()
print("Répartition :")
print(manifest["benchmark_split"].value_counts())

print()
print("Les 96 images ont été reconstruites.")
print("Les noms correspondent aux annotations CVAT.")
print("Les 48 images de calibration sont disponibles.")
print(" Les 48 images de test sont disponibles.")
print("Le dataset Training2 n'a pas été utilisé.")

In [ ]:
# CELLULE 4 — Vérité terrain, classes et fonctions communes
category_by_id = {category["id"]: category["name"] for category in coco["categories"]}
filename_by_image_id = {
    image["id"]: image["file_name"] for image in coco["images"]
}
frame_id_by_filename = dict(
    zip(manifest["frame_filename"], manifest["frame_id"])
)
split_by_frame_id = dict(
    zip(manifest["frame_id"], manifest["benchmark_split"])
)

gt_rows = []
for annotation in coco["annotations"]:
    x, y, width, height = map(float, annotation["bbox"])
    filename = filename_by_image_id[annotation["image_id"]]
    gt_rows.append({
        "frame_id": frame_id_by_filename[filename],
        "frame_filename": filename,
        "class_name": category_by_id[annotation["category_id"]],
        "x1": x,
        "y1": y,
        "x2": x + width,
        "y2": y + height,
    })
ground_truth_all = pd.DataFrame(gt_rows)

# La classe enfant est réservée à la proximité, pas au benchmark d'objets.
ground_truth = ground_truth_all[
    ground_truth_all["class_name"] != "child"
].reset_index(drop=True)
EVAL_CLASSES = sorted(ground_truth["class_name"].unique().tolist())
TEXT_LABELS = [name.replace("_", " ") for name in EVAL_CLASSES]
text_to_class = dict(zip(TEXT_LABELS, EVAL_CLASSES))

def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"^[0-9]+[:.) -]*", "", text)
    text = re.sub(r"^(a|an|the)\s+", "", text)
    text = text.replace("_", " ")
    text = re.sub(r"[^a-z0-9 ]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

normalized_prompt_to_class = {
    normalize_text(prompt): class_name
    for prompt, class_name in text_to_class.items()
}

def canonical_class(label):
    if isinstance(label, (int, np.integer)):
        index = int(label)
        return EVAL_CLASSES[index] if 0 <= index < len(EVAL_CLASSES) else None
    normalized = normalize_text(label)
    if normalized in normalized_prompt_to_class:
        return normalized_prompt_to_class[normalized]
    # Compatibilité avec les libellés textuels parfois reformulés.
    candidates = [
        (prompt, class_name)
        for prompt, class_name in normalized_prompt_to_class.items()
        if prompt in normalized or normalized in prompt
    ]
    if not candidates:
        return None
    return max(candidates, key=lambda item: len(item[0]))[1]

def box_iou_xyxy(box_a, box_b):
    ax1, ay1, ax2, ay2 = map(float, box_a)
    bx1, by1, bx2, by2 = map(float, box_b)
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    intersection = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0

def class_aware_nms(records, iou_threshold=NMS_IOU):
    kept = []
    by_class = defaultdict(list)
    for record in records:
        by_class[record["class_name"]].append(record)
    for _, class_records in by_class.items():
        remaining = sorted(
            class_records, key=lambda item: item["confidence"], reverse=True
        )
        while remaining:
            best = remaining.pop(0)
            kept.append(best)
            best_box = [best[k] for k in ("x1", "y1", "x2", "y2")]
            remaining = [
                candidate for candidate in remaining
                if box_iou_xyxy(
                    best_box,
                    [candidate[k] for k in ("x1", "y1", "x2", "y2")]
                ) < iou_threshold
            ]
    return kept

def synchronize():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def save_model_outputs(model_name, prediction_rows, latency_rows):
    prediction_columns = [
        "model", "frame_id", "frame_filename", "class_name",
        "confidence", "x1", "y1", "x2", "y2"
    ]
    predictions = pd.DataFrame(
        prediction_rows, columns=prediction_columns
    )
    latency = pd.DataFrame(latency_rows)
    predictions.to_csv(
        PREDICTION_DIR / f"{model_name}_predictions.csv", index=False
    )
    latency.to_csv(
        PREDICTION_DIR / f"{model_name}_latency.csv", index=False
    )
    print(
        f"{model_name}: {len(predictions)} prédictions, "
        f"{len(latency)} mesures de latence."
    )
    return predictions, latency

print("Annotations totales :", len(ground_truth_all))
print("Annotations objets  :", len(ground_truth))
print("Classes objets      :", len(EVAL_CLASSES))
print(EVAL_CLASSES)

## Inférence

Chaque cellule suivante charge **un seul modèle**, exécute les 96 images,
sauvegarde immédiatement les prédictions puis libère la mémoire GPU. Si une
session est interrompue, les CSV déjà produits peuvent être conservés dans
une version Kaggle avant de relancer.

Le seuil brut de 0,01 sert uniquement à conserver assez de candidats pour
la calibration. Ce n'est pas le seuil final.

In [ ]:
# CELLULE 5 — Grounding DINO
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor

MODEL_NAME = "grounding_dino"
MODEL_ID = "IDEA-Research/grounding-dino-base"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID)
model = model.to(DEVICE).eval()

def extract_processed_labels(result):
    if "text_labels" in result:
        return result["text_labels"]
    return result["labels"]

@torch.inference_mode()
def grounding_dino_predict(image):
    height, width = image.height, image.width
    inputs = processor(images=image, text=[TEXT_LABELS], return_tensors="pt")
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
    outputs = model(**inputs)
    kwargs = dict(
        outputs=outputs,
        target_sizes=[(height, width)],
        threshold=RAW_CONFIDENCE,
        text_threshold=RAW_CONFIDENCE,
    )
    try:
        result = processor.post_process_grounded_object_detection(
            input_ids=inputs.get("input_ids"),
            text_labels=[TEXT_LABELS],
            **kwargs,
        )[0]
    except TypeError:
        result = processor.post_process_grounded_object_detection(
            input_ids=inputs.get("input_ids"),
            **kwargs,
        )[0]

    records = []
    dropped = 0
    for box, score, label in zip(
        result["boxes"], result["scores"], extract_processed_labels(result)
    ):
        class_name = canonical_class(
            label.item() if torch.is_tensor(label) and label.ndim == 0 else label
        )
        if class_name is None:
            dropped += 1
            continue
        x1, y1, x2, y2 = map(float, box.detach().cpu().tolist())
        records.append({
            "class_name": class_name,
            "confidence": float(score.detach().cpu()),
            "x1": max(0.0, min(width, x1)),
            "y1": max(0.0, min(height, y1)),
            "x2": max(0.0, min(width, x2)),
            "y2": max(0.0, min(height, y2)),
        })
    return class_aware_nms(records), dropped

# Préchauffage hors mesure.
warmup_image = Image.open(FRAME_DIR / manifest.iloc[0]["frame_filename"]).convert("RGB")
grounding_dino_predict(warmup_image)

prediction_rows, latency_rows = [], []
dropped_total = 0
for position, row in enumerate(manifest.itertuples(index=False), start=1):
    image = Image.open(FRAME_DIR / row.frame_filename).convert("RGB")
    synchronize()
    start = time.perf_counter()
    records, dropped = grounding_dino_predict(image)
    synchronize()
    elapsed = time.perf_counter() - start
    dropped_total += dropped
    latency_rows.append({
        "model": MODEL_NAME,
        "frame_id": row.frame_id,
        "seconds": elapsed,
    })
    for record in records:
        prediction_rows.append({
            "model": MODEL_NAME,
            "frame_id": row.frame_id,
            "frame_filename": row.frame_filename,
            **record,
        })
    if position % 12 == 0:
        print(f"{position}/96")

grounding_dino_predictions, grounding_dino_latency = save_model_outputs(
    MODEL_NAME, prediction_rows, latency_rows
)
print("Libellés non résolus :", dropped_total)

del model, processor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# CELLULE 6 — YOLO-World
from ultralytics import YOLO

MODEL_NAME = "yolo_world"
model = YOLO("yolov8s-worldv2.pt")
model.set_classes(TEXT_LABELS)

def yolo_world_predict(image):
    result = model.predict(
        source=np.asarray(image),
        conf=RAW_CONFIDENCE,
        iou=0.99,          # NMS commune appliquée ensuite
        imgsz=640,
        max_det=500,
        device=0 if torch.cuda.is_available() else "cpu",
        verbose=False,
    )[0]
    records = []
    if result.boxes is None:
        return records, 0
    for box, score, label_index in zip(
        result.boxes.xyxy.detach().cpu().numpy(),
        result.boxes.conf.detach().cpu().numpy(),
        result.boxes.cls.detach().cpu().numpy(),
    ):
        label_index = int(label_index)
        raw_label = result.names[label_index]
        class_name = canonical_class(raw_label)
        if class_name is None:
            continue
        x1, y1, x2, y2 = map(float, box)
        records.append({
            "class_name": class_name,
            "confidence": float(score),
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
        })
    return class_aware_nms(records), 0

warmup_image = Image.open(FRAME_DIR / manifest.iloc[0]["frame_filename"]).convert("RGB")
yolo_world_predict(warmup_image)

prediction_rows, latency_rows = [], []
for position, row in enumerate(manifest.itertuples(index=False), start=1):
    image = Image.open(FRAME_DIR / row.frame_filename).convert("RGB")
    synchronize()
    start = time.perf_counter()
    records, _ = yolo_world_predict(image)
    synchronize()
    elapsed = time.perf_counter() - start
    latency_rows.append({
        "model": MODEL_NAME,
        "frame_id": row.frame_id,
        "seconds": elapsed,
    })
    for record in records:
        prediction_rows.append({
            "model": MODEL_NAME,
            "frame_id": row.frame_id,
            "frame_filename": row.frame_filename,
            **record,
        })
    if position % 12 == 0:
        print(f"{position}/96")

yolo_world_predictions, yolo_world_latency = save_model_outputs(
    MODEL_NAME, prediction_rows, latency_rows
)

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# CELLULE 7 — OWLv2
from transformers import Owlv2ForObjectDetection, Owlv2Processor

MODEL_NAME = "owlv2"
MODEL_ID = "google/owlv2-base-patch16-ensemble"
processor = Owlv2Processor.from_pretrained(MODEL_ID)
model = Owlv2ForObjectDetection.from_pretrained(MODEL_ID)
model = model.to(DEVICE).eval()

@torch.inference_mode()
def owlv2_predict(image):
    height, width = image.height, image.width
    inputs = processor(text=[TEXT_LABELS], images=image, return_tensors="pt")
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
    outputs = model(**inputs)
    target_sizes = torch.tensor([(height, width)], device=DEVICE)
    try:
        result = processor.post_process_grounded_object_detection(
            outputs=outputs,
            target_sizes=target_sizes,
            threshold=RAW_CONFIDENCE,
            text_labels=[TEXT_LABELS],
        )[0]
    except (AttributeError, TypeError):
        result = processor.post_process_object_detection(
            outputs=outputs,
            target_sizes=target_sizes,
            threshold=RAW_CONFIDENCE,
        )[0]

    labels = (
        result["text_labels"]
        if "text_labels" in result
        else result["labels"]
    )
    records, dropped = [], 0
    for box, score, label in zip(result["boxes"], result["scores"], labels):
        raw_label = (
            label.item()
            if torch.is_tensor(label) and label.ndim == 0
            else label
        )
        class_name = canonical_class(raw_label)
        if class_name is None:
            dropped += 1
            continue
        x1, y1, x2, y2 = map(float, box.detach().cpu().tolist())
        records.append({
            "class_name": class_name,
            "confidence": float(score.detach().cpu()),
            "x1": max(0.0, min(width, x1)),
            "y1": max(0.0, min(height, y1)),
            "x2": max(0.0, min(width, x2)),
            "y2": max(0.0, min(height, y2)),
        })
    return class_aware_nms(records), dropped

warmup_image = Image.open(FRAME_DIR / manifest.iloc[0]["frame_filename"]).convert("RGB")
owlv2_predict(warmup_image)

prediction_rows, latency_rows = [], []
dropped_total = 0
for position, row in enumerate(manifest.itertuples(index=False), start=1):
    image = Image.open(FRAME_DIR / row.frame_filename).convert("RGB")
    synchronize()
    start = time.perf_counter()
    records, dropped = owlv2_predict(image)
    synchronize()
    elapsed = time.perf_counter() - start
    dropped_total += dropped
    latency_rows.append({
        "model": MODEL_NAME,
        "frame_id": row.frame_id,
        "seconds": elapsed,
    })
    for record in records:
        prediction_rows.append({
            "model": MODEL_NAME,
            "frame_id": row.frame_id,
            "frame_filename": row.frame_filename,
            **record,
        })
    if position % 12 == 0:
        print(f"{position}/96")

owlv2_predictions, owlv2_latency = save_model_outputs(
    MODEL_NAME, prediction_rows, latency_rows
)
print("Libellés non résolus :", dropped_total)

del model, processor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# CELLULE 7B — YOLOE-26S
# DÉTECTION OPEN-VOCABULARY
# ============================================================

import gc
import time
import numpy as np
import torch
from PIL import Image
from ultralytics import YOLOE


MODEL_NAME = "yoloe_26s"
MODEL_ID = "yoloe-26s-seg.pt"

print("Chargement de :", MODEL_ID)

model = YOLOE(MODEL_ID)

# Même vocabulaire que pour tous les autres détecteurs
model.set_classes(TEXT_LABELS)


def yoloe_predict(image):
    """
    Exécute YOLOE-26s sur une image et renvoie les bounding boxes
    dans le format commun utilisé par le benchmark.
    """

    result = model.predict(
        source=np.asarray(image),
        conf=RAW_CONFIDENCE,
        iou=0.99,  # La NMS commune sera appliquée ensuite
        imgsz=640,
        max_det=500,
        agnostic_nms=False,
        device=0 if torch.cuda.is_available() else "cpu",
        verbose=False,
    )[0]

    records = []
    dropped = 0

    if result.boxes is None:
        return records, dropped

    boxes = result.boxes.xyxy.detach().cpu().numpy()
    scores = result.boxes.conf.detach().cpu().numpy()
    label_indices = result.boxes.cls.detach().cpu().numpy()

    for box, score, label_index in zip(
        boxes,
        scores,
        label_indices,
    ):
        label_index = int(label_index)
        raw_label = result.names[label_index]
        class_name = canonical_class(raw_label)

        if class_name is None:
            dropped += 1
            continue

        x1, y1, x2, y2 = map(float, box)

        # Suppression immédiate des boîtes géométriquement impossibles
        if x2 <= x1 or y2 <= y1:
            dropped += 1
            continue

        records.append(
            {
                "class_name": class_name,
                "confidence": float(score),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
            }
        )

    records = class_aware_nms(
        records,
        iou_threshold=NMS_IOU,
    )

    return records, dropped


# ------------------------------------------------------------
# Préchauffage : cette mesure n'est pas comptée dans la latence
# ------------------------------------------------------------

warmup_image = Image.open(
    FRAME_DIR / manifest.iloc[0]["frame_filename"]
).convert("RGB")

yoloe_predict(warmup_image)

print("Préchauffage terminé.")


# ------------------------------------------------------------
# Inférence sur les 96 images
# ------------------------------------------------------------

prediction_rows = []
latency_rows = []
dropped_total = 0

for position, row in enumerate(
    manifest.itertuples(index=False),
    start=1,
):
    image = Image.open(
        FRAME_DIR / row.frame_filename
    ).convert("RGB")

    synchronize()
    start_time = time.perf_counter()

    records, dropped = yoloe_predict(image)

    synchronize()
    elapsed = time.perf_counter() - start_time

    dropped_total += dropped

    latency_rows.append(
        {
            "model": MODEL_NAME,
            "frame_id": row.frame_id,
            "seconds": elapsed,
        }
    )

    for record in records:
        prediction_rows.append(
            {
                "model": MODEL_NAME,
                "frame_id": row.frame_id,
                "frame_filename": row.frame_filename,
                **record,
            }
        )

    if position % 12 == 0:
        print(f"{position}/96 images traitées")


yoloe_predictions, yoloe_latency = save_model_outputs(
    MODEL_NAME,
    prediction_rows,
    latency_rows,
)

print("Libellés ou boîtes supprimés :", dropped_total)
print(" YOLOE-26s terminé.")


# ------------------------------------------------------------
# Libération de la mémoire
# ------------------------------------------------------------

del model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# CELLULE 7C — OMDET-TURBO SWIN-TINY
# DÉTECTION OPEN-VOCABULARY
# ============================================================

import gc
import time
import torch
from PIL import Image
from transformers import (
    AutoProcessor,
    OmDetTurboForObjectDetection,
)


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

MODEL_NAME = "omdet_turbo"
MODEL_ID = "omlab/omdet-turbo-swin-tiny-hf"

OMDET_DEVICE = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("Chargement de :", MODEL_ID)
print("Périphérique cible :", OMDET_DEVICE)


# ------------------------------------------------------------
# 2. NETTOYAGE DE LA MÉMOIRE
# ------------------------------------------------------------

# Supprime les modèles éventuellement laissés par un essai précédent.
for variable_name in ["model", "processor"]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


# ------------------------------------------------------------
# 3. CHARGEMENT DU PROCESSOR
# ------------------------------------------------------------

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
)


# ------------------------------------------------------------
# 4. CHARGEMENT SÛR DU MODÈLE
# ------------------------------------------------------------

# Le modèle est d'abord entièrement chargé en mémoire CPU.
#
# device_map=None :
#     empêche Accelerate de répartir automatiquement le modèle.
#
# low_cpu_mem_usage=False :
#     empêche l'initialisation de certains composants du backbone
#     Swin sur le périphérique virtuel "meta".
#
# Le transfert vers le GPU sera réalisé seulement après avoir
# vérifié que tous les paramètres et buffers existent réellement.

model = OmDetTurboForObjectDetection.from_pretrained(
    MODEL_ID,
    device_map=None,
    low_cpu_mem_usage=False,
    dtype=torch.float32,
).eval()


# ------------------------------------------------------------
# 5. RECONSTRUCTION DES BUFFERS SWIN RESTÉS SUR META
# ------------------------------------------------------------

reconstructed_masks = []

for module_name, module in model.named_modules():
    attention_mask = getattr(
        module,
        "attn_mask",
        None,
    )

    if (
        torch.is_tensor(attention_mask)
        and attention_mask.is_meta
    ):
        if not hasattr(module, "get_attn_mask"):
            raise RuntimeError(
                f"Impossible de reconstruire le masque de "
                f"{module_name} : méthode get_attn_mask absente."
            )

        if not hasattr(module, "norm1"):
            raise RuntimeError(
                f"Impossible de déterminer le périphérique "
                f"du bloc {module_name}."
            )

        # Les poids du bloc sont actuellement sur CPU.
        target_device = module.norm1.weight.device
        target_dtype = module.norm1.weight.dtype

        # Reconstruction par la méthode native du bloc Swin.
        rebuilt_mask = module.get_attn_mask(
            device=target_device,
            dtype=target_dtype,
        )

        if rebuilt_mask is None:
            raise RuntimeError(
                f"La reconstruction du masque de "
                f"{module_name} a renvoyé None."
            )

        if rebuilt_mask.is_meta:
            raise RuntimeError(
                f"Le masque reconstruit de {module_name} "
                f"est toujours sur meta."
            )

        # L'affectation conserve son statut de buffer
        # non persistant dans le module PyTorch.
        module.attn_mask = rebuilt_mask

        reconstructed_masks.append(
            {
                "module": module_name,
                "shape": tuple(rebuilt_mask.shape),
                "device": str(rebuilt_mask.device),
                "dtype": str(rebuilt_mask.dtype),
            }
        )


print(
    "Masques Swin reconstruits :",
    len(reconstructed_masks),
)

for mask_information in reconstructed_masks:
    print(
        " -",
        mask_information["module"],
        "| shape :",
        mask_information["shape"],
        "| device :",
        mask_information["device"],
    )


# ------------------------------------------------------------
# VÉRIFICATION APRÈS RECONSTRUCTION
# ------------------------------------------------------------

meta_parameters = [
    name
    for name, parameter in model.named_parameters()
    if parameter.is_meta
]

meta_buffers = [
    name
    for name, buffer in model.named_buffers()
    if buffer.is_meta
]

print("Paramètres meta restants :", len(meta_parameters))
print("Buffers meta restants    :", len(meta_buffers))

assert not meta_parameters, (
    "Des paramètres du modèle sont encore sur meta : "
    f"{meta_parameters[:10]}"
)

assert not meta_buffers, (
    "Des buffers du modèle sont encore sur meta : "
    f"{meta_buffers[:10]}"
)

assert len(reconstructed_masks) == 6, (
    "Six masques Swin étaient attendus, mais "
    f"{len(reconstructed_masks)} ont été reconstruits."
)

print("Reconstruction des buffers Swin : réussie")

# ------------------------------------------------------------
# 6. TRANSFERT VERS LE GPU
# ------------------------------------------------------------

model = model.to(OMDET_DEVICE)
model.eval()

print("OmDet-Turbo correctement chargé sur :", OMDET_DEVICE)


# ------------------------------------------------------------
# 7. FONCTION D'INFÉRENCE
# ------------------------------------------------------------

@torch.inference_mode()
def omdet_predict(image):
    """
    Exécute OmDet-Turbo sur une image PIL et convertit les
    résultats dans le format commun utilisé par le benchmark.

    Retourne
    --------
    records : list[dict]
        Détections conservées après nettoyage et NMS.
    dropped : int
        Nombre de détections supprimées parce que le libellé
        était inconnu ou que la bounding box était invalide.
    """

    width, height = image.size

    # OmDet-Turbo reçoit l'image et les catégories textuelles
    # que le modèle doit rechercher.
    inputs = processor(
        images=image,
        text=TEXT_LABELS,
        return_tensors="pt",
    )

    # Transfert de tous les tenseurs d'entrée sur le même
    # périphérique que le modèle.
    inputs = {
        key: value.to(OMDET_DEVICE)
        if torch.is_tensor(value)
        else value
        for key, value in inputs.items()
    }

    outputs = model(**inputs)

    # Conversion des sorties brutes en bounding boxes exprimées
    # dans les dimensions originales de l'image.
    result = processor.post_process_grounded_object_detection(
        outputs,
        target_sizes=[(height, width)],
        text_labels=TEXT_LABELS,
        threshold=RAW_CONFIDENCE,
        nms_threshold=0.99,
        max_num_det=500,
    )[0]

    records = []
    dropped = 0

    # Selon la version de Transformers, les classes peuvent
    # apparaître sous "text_labels" ou sous "labels".
    if "text_labels" in result:
        labels = result["text_labels"]
    else:
        labels = result["labels"]

    for box, score, label in zip(
        result["boxes"],
        result["scores"],
        labels,
    ):
        # Conversion d'un éventuel tenseur scalaire.
        if torch.is_tensor(label) and label.ndim == 0:
            raw_label = label.item()
        else:
            raw_label = label

        # Conversion vers le nom de classe commun au benchmark.
        class_name = canonical_class(raw_label)

        if class_name is None:
            dropped += 1
            continue

        x1, y1, x2, y2 = map(
            float,
            box.detach().cpu().tolist(),
        )

        # Limitation des coordonnées aux dimensions de l'image.
        x1 = max(0.0, min(float(width), x1))
        y1 = max(0.0, min(float(height), y1))
        x2 = max(0.0, min(float(width), x2))
        y2 = max(0.0, min(float(height), y2))

        # Suppression des boîtes de largeur ou hauteur nulle.
        if x2 <= x1 or y2 <= y1:
            dropped += 1
            continue

        records.append(
            {
                "class_name": class_name,
                "confidence": float(
                    score.detach().cpu().item()
                ),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
            }
        )

    # La NMS interne est volontairement presque désactivée
    # avec 0.99. La même NMS est ensuite appliquée à tous les
    # modèles afin de conserver une comparaison équitable.
    records = class_aware_nms(
        records,
        iou_threshold=NMS_IOU,
    )

    return records, dropped


# ------------------------------------------------------------
# 8. PRÉCHAUFFAGE NON COMPTABILISÉ
# ------------------------------------------------------------

warmup_path = (
    FRAME_DIR
    / manifest.iloc[0]["frame_filename"]
)

warmup_image = Image.open(
    warmup_path
).convert("RGB")

_ = omdet_predict(warmup_image)

if torch.cuda.is_available():
    torch.cuda.synchronize()

print("Préchauffage terminé.")


# ------------------------------------------------------------
# 9. INFÉRENCE SUR LES 96 IMAGES
# ------------------------------------------------------------

prediction_rows = []
latency_rows = []
dropped_total = 0

number_of_images = len(manifest)

for position, row in enumerate(
    manifest.itertuples(index=False),
    start=1,
):
    image_path = FRAME_DIR / row.frame_filename

    image = Image.open(
        image_path
    ).convert("RGB")

    # Synchronisation avant et après l'inférence pour mesurer
    # correctement le temps de calcul GPU.
    synchronize()
    start_time = time.perf_counter()

    records, dropped = omdet_predict(image)

    synchronize()
    elapsed = time.perf_counter() - start_time

    dropped_total += dropped

    latency_rows.append(
        {
            "model": MODEL_NAME,
            "frame_id": row.frame_id,
            "seconds": elapsed,
        }
    )

    for record in records:
        prediction_rows.append(
            {
                "model": MODEL_NAME,
                "frame_id": row.frame_id,
                "frame_filename": row.frame_filename,
                **record,
            }
        )

    if (
        position % 12 == 0
        or position == number_of_images
    ):
        print(
            f"{position}/{number_of_images} images traitées"
        )


# ------------------------------------------------------------
# 10. SAUVEGARDE DES RÉSULTATS
# ------------------------------------------------------------

omdet_predictions, omdet_latency = save_model_outputs(
    MODEL_NAME,
    prediction_rows,
    latency_rows,
)

print()
print("Détections conservées :", len(omdet_predictions))
print("Libellés ou boîtes supprimés :", dropped_total)
print("OmDet-Turbo terminé.")


# ------------------------------------------------------------
# 11. TESTS DE VALIDITÉ DES SORTIES
# ------------------------------------------------------------

assert len(omdet_latency) == number_of_images, (
    "Il doit exister exactement une mesure de latence "
    "par image."
)

assert (
    omdet_latency["seconds"] > 0
).all(), "Certaines latences sont invalides."

if not omdet_predictions.empty:
    assert (
        omdet_predictions["confidence"].between(0, 1)
    ).all(), "Certaines confiances sont invalides."

    assert (
        omdet_predictions["x2"]
        > omdet_predictions["x1"]
    ).all(), "Certaines bounding boxes ont une largeur invalide."

    assert (
        omdet_predictions["y2"]
        > omdet_predictions["y1"]
    ).all(), "Certaines bounding boxes ont une hauteur invalide."

    # Les prédictions ont été normalisées par canonical_class().
# Il faut donc également normaliser les classes attendues avant
# d'effectuer la comparaison.

expected_canonical_classes = set()

for text_label in TEXT_LABELS:
    canonical_label = canonical_class(text_label)

    if canonical_label is not None:
        expected_canonical_classes.add(canonical_label)

predicted_classes = set(
    omdet_predictions["class_name"].unique()
)

unexpected_classes = (
    predicted_classes
    - expected_canonical_classes
)

print(
    "Classes attendues après normalisation :",
    sorted(expected_canonical_classes),
)

print(
    "Classes prédites :",
    sorted(predicted_classes),
)

print(
    "Classes réellement inattendues :",
    sorted(unexpected_classes),
)

assert not unexpected_classes, (
    "OmDet-Turbo a produit des classes réellement "
    f"étrangères au benchmark : {sorted(unexpected_classes)}"
)

print("Tests de validité OmDet-Turbo : réussis")


# ------------------------------------------------------------
# 12. LIBÉRATION DE LA MÉMOIRE
# ------------------------------------------------------------

del model
del processor
del warmup_image

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Mémoire OmDet-Turbo libérée.")

In [ ]:
# CELLULE 8 — Chargement sorties et fonctions d'évaluation
MODEL_NAMES = [
    "grounding_dino",
    "yolo_world",
    "owlv2",
    "yoloe_26s",
    "omdet_turbo",
]

prediction_frames = []
latency_frames = []
prediction_columns = [
    "model", "frame_id", "frame_filename", "class_name",
    "confidence", "x1", "y1", "x2", "y2"
]
for model_name in MODEL_NAMES:
    prediction_path = PREDICTION_DIR / f"{model_name}_predictions.csv"
    latency_path = PREDICTION_DIR / f"{model_name}_latency.csv"
    assert prediction_path.exists(), f"Sortie absente : {prediction_path}"
    assert latency_path.exists(), f"Sortie absente : {latency_path}"
    predictions = pd.read_csv(prediction_path)
    if predictions.empty:
        predictions = pd.DataFrame(columns=prediction_columns)
    prediction_frames.append(predictions)
    latency_frames.append(pd.read_csv(latency_path))

all_predictions = pd.concat(prediction_frames, ignore_index=True)
all_latency = pd.concat(latency_frames, ignore_index=True)
all_predictions["frame_id"] = all_predictions["frame_id"].astype(
    manifest["frame_id"].dtype
)

def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else 0.0

def match_frame(gt_frame, pred_frame, iou_threshold=MATCH_IOU):
    gt_records = gt_frame.to_dict("records")
    pred_records = pred_frame.sort_values(
        "confidence", ascending=False
    ).to_dict("records")
    used_gt = set()
    matches = []
    prediction_is_tp = []

    for prediction in pred_records:
        candidate_indices = [
            index for index, truth in enumerate(gt_records)
            if index not in used_gt
            and truth["class_name"] == prediction["class_name"]
        ]
        best_index, best_iou = None, 0.0
        pred_box = [prediction[k] for k in ("x1", "y1", "x2", "y2")]
        for index in candidate_indices:
            truth = gt_records[index]
            truth_box = [truth[k] for k in ("x1", "y1", "x2", "y2")]
            iou = box_iou_xyxy(pred_box, truth_box)
            if iou > best_iou:
                best_index, best_iou = index, iou
        is_tp = best_index is not None and best_iou >= iou_threshold
        prediction_is_tp.append(int(is_tp))
        matches.append((best_index if is_tp else None, best_iou))
        if is_tp:
            used_gt.add(best_index)

    return {
        "tp": sum(prediction_is_tp),
        "fp": len(pred_records) - sum(prediction_is_tp),
        "fn": len(gt_records) - len(used_gt),
        "prediction_is_tp": prediction_is_tp,
        "matches": matches,
    }

def evaluate_counts(predictions, split, threshold, class_filter=None):
    frame_ids = manifest.loc[
        manifest["benchmark_split"] == split, "frame_id"
    ].tolist()
    gt_split = ground_truth[ground_truth["frame_id"].isin(frame_ids)]
    pred_split = predictions[
        predictions["frame_id"].isin(frame_ids)
        & (predictions["confidence"] >= threshold)
    ]
    if class_filter is not None:
        gt_split = gt_split[gt_split["class_name"] == class_filter]
        pred_split = pred_split[pred_split["class_name"] == class_filter]

    tp = fp = fn = 0
    frames_without_detection = 0
    for frame_id in frame_ids:
        gt_frame = gt_split[gt_split["frame_id"] == frame_id]
        pred_frame = pred_split[pred_split["frame_id"] == frame_id]
        if pred_frame.empty:
            frames_without_detection += 1
        result = match_frame(gt_frame, pred_frame)
        tp += result["tp"]
        fp += result["fp"]
        fn += result["fn"]
    precision = safe_divide(tp, tp + fp)
    recall = safe_divide(tp, tp + fn)
    f1 = safe_divide(2 * precision * recall, precision + recall)
    return {
        "tp": tp, "fp": fp, "fn": fn,
        "precision": precision, "recall": recall, "f1": f1,
        "fp_per_frame": fp / len(frame_ids),
        "frames_without_detection": frames_without_detection,
    }

def average_precision_50(predictions, split, class_name):
    frame_ids = manifest.loc[
        manifest["benchmark_split"] == split, "frame_id"
    ].tolist()
    gt_class = ground_truth[
        ground_truth["frame_id"].isin(frame_ids)
        & (ground_truth["class_name"] == class_name)
    ]
    number_gt = len(gt_class)
    if number_gt == 0:
        return np.nan

    pred_class = predictions[
        predictions["frame_id"].isin(frame_ids)
        & (predictions["class_name"] == class_name)
    ].sort_values("confidence", ascending=False)

    gt_by_frame = {
        frame_id: frame.to_dict("records")
        for frame_id, frame in gt_class.groupby("frame_id")
    }
    used = defaultdict(set)
    tp_flags, fp_flags = [], []
    for prediction in pred_class.to_dict("records"):
        truths = gt_by_frame.get(prediction["frame_id"], [])
        best_index, best_iou = None, 0.0
        pred_box = [prediction[k] for k in ("x1", "y1", "x2", "y2")]
        for index, truth in enumerate(truths):
            if index in used[prediction["frame_id"]]:
                continue
            truth_box = [truth[k] for k in ("x1", "y1", "x2", "y2")]
            iou = box_iou_xyxy(pred_box, truth_box)
            if iou > best_iou:
                best_index, best_iou = index, iou
        is_tp = best_index is not None and best_iou >= MATCH_IOU
        tp_flags.append(int(is_tp))
        fp_flags.append(int(not is_tp))
        if is_tp:
            used[prediction["frame_id"]].add(best_index)

    if not tp_flags:
        return 0.0
    cumulative_tp = np.cumsum(tp_flags)
    cumulative_fp = np.cumsum(fp_flags)
    recall = cumulative_tp / number_gt
    precision = cumulative_tp / np.maximum(
        cumulative_tp + cumulative_fp, 1
    )
    # Interpolation COCO à 101 points, pour IoU=0,50.
    return float(np.mean([
        np.max(precision[recall >= level])
        if np.any(recall >= level) else 0.0
        for level in np.linspace(0, 1, 101)
    ]))

def per_class_metrics(predictions, split, threshold):
    rows = []
    for class_name in EVAL_CLASSES:
        counts = evaluate_counts(
            predictions, split, threshold, class_filter=class_name
        )
        support = len(ground_truth[
            (ground_truth["class_name"] == class_name)
            & ground_truth["frame_id"].isin(
                manifest.loc[
                    manifest["benchmark_split"] == split, "frame_id"
                ]
            )
        ])
        if support == 0:
            continue
        rows.append({
            "class_name": class_name,
            "support": support,
            **counts,
            "ap50": average_precision_50(
                predictions, split, class_name
            ),
        })
    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# NETTOYAGE DES BOUNDING BOXES INVALIDES
# ============================================================

coordinate_columns = ["x1", "y1", "x2", "y2"]

# Conversion explicite en nombres
for column in coordinate_columns + ["confidence"]:
    all_predictions[column] = pd.to_numeric(
        all_predictions[column],
        errors="coerce",
    )

# Détection des valeurs absentes ou infinies
non_finite_mask = ~np.isfinite(
    all_predictions[
        coordinate_columns + ["confidence"]
    ].to_numpy()
).all(axis=1)

# Une boîte valide doit avoir une largeur et une hauteur strictement positives
invalid_geometry_mask = (
    (all_predictions["x2"] <= all_predictions["x1"])
    | (all_predictions["y2"] <= all_predictions["y1"])
)

invalid_mask = non_finite_mask | invalid_geometry_mask

invalid_predictions = all_predictions.loc[invalid_mask].copy()

print("Prédictions totales avant nettoyage :", len(all_predictions))
print("Bounding boxes invalides             :", len(invalid_predictions))

if not invalid_predictions.empty:

    print("\nRépartition par modèle :")
    print(
        invalid_predictions["model"]
        .value_counts()
        .to_string()
    )

    display(
        invalid_predictions[
            [
                "model",
                "frame_id",
                "frame_filename",
                "class_name",
                "confidence",
                "x1",
                "y1",
                "x2",
                "y2",
            ]
        ]
    )

    # Conservation d'une trace pour assurer la transparence du traitement
    invalid_predictions.to_csv(
        OUTPUT_DIR / "invalid_predictions_removed.csv",
        index=False,
    )

# Suppression des détections géométriquement impossibles
all_predictions = (
    all_predictions.loc[~invalid_mask]
    .copy()
    .reset_index(drop=True)
)

all_predictions.to_csv(
    OUTPUT_DIR / "all_predictions_cleaned.csv",
    index=False,
)

print("\nPrédictions conservées :", len(all_predictions))

assert np.isfinite(
    all_predictions[
        coordinate_columns + ["confidence"]
    ].to_numpy()
).all()

assert (
    all_predictions["x2"] > all_predictions["x1"]
).all()

assert (
    all_predictions["y2"] > all_predictions["y1"]
).all()

print(" Nettoyage des bounding boxes terminé.")

In [ ]:
# CELLULE 9 — Tests unitaires et tests d'intégration AVANT l'évaluation
# Tests unitaires : IoU, NMS et appariement.
assert math.isclose(box_iou_xyxy([0, 0, 10, 10], [0, 0, 10, 10]), 1.0)
assert math.isclose(box_iou_xyxy([0, 0, 10, 10], [20, 20, 30, 30]), 0.0)
assert math.isclose(
    box_iou_xyxy([0, 0, 10, 10], [5, 0, 15, 10]), 1 / 3
)

nms_example = [
    {"class_name": "chair", "confidence": 0.9, "x1": 0, "y1": 0, "x2": 10, "y2": 10},
    {"class_name": "chair", "confidence": 0.8, "x1": 1, "y1": 1, "x2": 10, "y2": 10},
    {"class_name": "table", "confidence": 0.7, "x1": 1, "y1": 1, "x2": 10, "y2": 10},
]
assert len(class_aware_nms(nms_example, 0.5)) == 2

gt_example = pd.DataFrame([
    {"class_name": "chair", "x1": 0, "y1": 0, "x2": 10, "y2": 10}
])
pred_example = pd.DataFrame([
    {"class_name": "chair", "confidence": 0.9, "x1": 0, "y1": 0, "x2": 10, "y2": 10},
    {"class_name": "chair", "confidence": 0.8, "x1": 0, "y1": 0, "x2": 10, "y2": 10},
])
match_example = match_frame(gt_example, pred_example)
assert (match_example["tp"], match_example["fp"], match_example["fn"]) == (1, 1, 0)

# Tests d'intégration des données et des sorties.
calibration_videos = set(
    manifest.loc[manifest["benchmark_split"] == "calibration", "video_id"]
)
test_videos = set(
    manifest.loc[manifest["benchmark_split"] == "test", "video_id"]
)
assert calibration_videos.isdisjoint(test_videos), (
    "Fuite de données : une vidéo apparaît en calibration et en test."
)
assert len(calibration_videos) == 12 and len(test_videos) == 12
assert ground_truth["class_name"].isin(EVAL_CLASSES).all()
assert not (ground_truth["class_name"] == "child").any()

expected_frame_ids = set(manifest["frame_id"])
for model_name in MODEL_NAMES:
    model_latency = all_latency[all_latency["model"] == model_name]
    model_predictions = all_predictions[
        all_predictions["model"] == model_name
    ]
    assert len(model_latency) == 96
    assert set(model_latency["frame_id"]) == expected_frame_ids
    assert model_predictions["class_name"].isin(EVAL_CLASSES).all()
    if not model_predictions.empty:
        assert (model_predictions["confidence"].between(0, 1)).all()
        assert (model_predictions["x2"] > model_predictions["x1"]).all()
        assert (model_predictions["y2"] > model_predictions["y1"]).all()

print("Tests unitaires : réussis")
print(" Tests d'intégration : réussis")

In [ ]:
# CELLULE 10 — Calibration indépendante des seuils
calibration_rows = []
selected_thresholds = {}

for model_name in MODEL_NAMES:
    model_predictions = all_predictions[
        all_predictions["model"] == model_name
    ]
    model_rows = []
    for threshold in THRESHOLD_GRID:
        metrics = evaluate_counts(
            model_predictions, "calibration", float(threshold)
        )
        model_rows.append({
            "model": model_name,
            "threshold": float(threshold),
            **metrics,
        })
    calibration_rows.extend(model_rows)
    best = max(
        model_rows,
        key=lambda row: (
            row["f1"],
            row["recall"],
            row["precision"],
            -row["threshold"],
        ),
    )
    selected_thresholds[model_name] = best["threshold"]

calibration_results = pd.DataFrame(calibration_rows)
calibration_results.to_csv(
    OUTPUT_DIR / "threshold_calibration_curves.csv", index=False
)
selected_threshold_table = pd.DataFrame([
    {
        "model": model_name,
        "selected_threshold": threshold,
        **evaluate_counts(
            all_predictions[all_predictions["model"] == model_name],
            "calibration",
            threshold,
        ),
    }
    for model_name, threshold in selected_thresholds.items()
])
selected_threshold_table.to_csv(
    OUTPUT_DIR / "selected_thresholds.csv", index=False
)

display(selected_threshold_table.round(4))

plt.figure(figsize=(10, 6))
for model_name in MODEL_NAMES:
    subset = calibration_results[
        calibration_results["model"] == model_name
    ]
    plt.plot(
        subset["threshold"], subset["f1"],
        label=f"{model_name} — seuil={selected_thresholds[model_name]:.2f}"
    )
plt.xlabel("Seuil de confiance")
plt.ylabel("F1 micro sur la calibration")
plt.title("Choix des seuils sur le lot de calibration")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "threshold_calibration.png", dpi=180)
plt.show()

In [ ]:
# CELLULE 11 — Évaluation finale sur le test et latence
final_rows = []
per_class_frames = []

for model_name in MODEL_NAMES:
    model_predictions = all_predictions[
        all_predictions["model"] == model_name
    ]
    threshold = selected_thresholds[model_name]
    micro = evaluate_counts(model_predictions, "test", threshold)
    per_class = per_class_metrics(model_predictions, "test", threshold)
    per_class.insert(0, "model", model_name)
    per_class_frames.append(per_class)

    latency_ms = (
        all_latency.loc[
            all_latency["model"] == model_name, "seconds"
        ].to_numpy() * 1000
    )
    final_rows.append({
        "model": model_name,
        "selected_threshold": threshold,
        **micro,
        "macro_precision": per_class["precision"].mean(),
        "macro_recall": per_class["recall"].mean(),
        "macro_f1": per_class["f1"].mean(),
        "map50": per_class["ap50"].mean(),
        "latency_mean_ms": latency_ms.mean(),
        "latency_median_ms": np.median(latency_ms),
        "latency_p95_ms": np.percentile(latency_ms, 95),
        "fps_from_mean": 1000 / latency_ms.mean(),
    })

final_results = pd.DataFrame(final_rows).sort_values(
    ["f1", "recall", "latency_mean_ms"],
    ascending=[False, False, True],
).reset_index(drop=True)
per_class_results = pd.concat(per_class_frames, ignore_index=True)

final_results.to_csv(OUTPUT_DIR / "final_test_metrics.csv", index=False)
per_class_results.to_csv(
    OUTPUT_DIR / "final_per_class_metrics.csv", index=False
)

display_columns = [
    "model", "selected_threshold", "tp", "fp", "fn",
    "precision", "recall", "f1", "macro_f1", "map50",
    "fp_per_frame", "frames_without_detection",
    "latency_mean_ms", "latency_p95_ms", "fps_from_mean"
]
display(final_results[display_columns].round(4))

best_model = final_results.iloc[0]["model"]
print(
    f"Modèle classé premier selon F1, puis rappel, puis latence : {best_model}"
)

In [ ]:
# CELLULE 12 — Graphiques destinés au rapport
long_metrics = final_results.melt(
    id_vars="model",
    value_vars=["precision", "recall", "f1", "map50"],
    var_name="metric",
    value_name="value",
)
plt.figure(figsize=(10, 6))
sns.barplot(data=long_metrics, x="model", y="value", hue="metric")
plt.ylim(0, 1)
plt.ylabel("Score")
plt.xlabel("Modèle")
plt.title("Comparaison finale sur les 48 images de test")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "final_detection_metrics.png", dpi=180)
plt.show()

plt.figure(figsize=(8, 5))
sns.barplot(
    data=final_results,
    x="model",
    y="latency_mean_ms",
    color="#4C78A8",
)
plt.ylabel("Latence moyenne par image (ms)")
plt.xlabel("Modèle")
plt.title("Coût de traitement — lot de taille 1")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "final_latency.png", dpi=180)
plt.show()

f1_pivot = per_class_results.pivot(
    index="class_name", columns="model", values="f1"
)
plt.figure(figsize=(8, max(7, 0.30 * len(f1_pivot))))
sns.heatmap(f1_pivot, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1)
plt.title("F1 par classe sur le lot de test")
plt.xlabel("Modèle")
plt.ylabel("Classe")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "per_class_f1_heatmap.png", dpi=180)
plt.show()

In [ ]:
# ============================================================
# CELLULE 13 — EXEMPLES QUALITATIFS
# ============================================================

from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import numpy as np


COLORS = {
    "ground_truth": "#00C853",
    "grounding_dino": "#FF6D00",
    "yolo_world": "#2962FF",
    "owlv2": "#AA00FF",
    "yoloe_26s": "#D81B60",
    "omdet_turbo": "#00ACC1",
}


def draw_boxes(
    image,
    records,
    color,
    show_confidence,
):
    """
    Dessine les bounding boxes et leurs classes sur une image.
    """

    rendered = image.copy()
    drawer = ImageDraw.Draw(rendered)

    for record in records:
        box = [
            record[key]
            for key in ("x1", "y1", "x2", "y2")
        ]

        drawer.rectangle(
            box,
            outline=color,
            width=3,
        )

        label = str(record["class_name"])

        if show_confidence:
            label += f" {record['confidence']:.2f}"

        x1, y1, _, _ = box

        drawer.text(
            (x1 + 2, max(0, y1 - 13)),
            label,
            fill=color,
        )

    return rendered


# ------------------------------------------------------------
# Sélection de six images du lot de test
# ------------------------------------------------------------

test_manifest = (
    manifest[
        manifest["benchmark_split"] == "test"
    ]
    .sort_values(
        ["class_name", "frame_id"]
    )
)

example_rows = (
    test_manifest
    .groupby(
        "class_name",
        as_index=False,
    )
    .first()
    .head(50)
)

print(
    "Nombre d'exemples qualitatifs :",
    len(example_rows),
)


# ------------------------------------------------------------
# Génération des figures
# ------------------------------------------------------------

for row in example_rows.itertuples(index=False):
    image_path = (
        FRAME_DIR
        / row.frame_filename
    )

    image = Image.open(
        image_path
    ).convert("RGB")

    panels = []

    # Vérité terrain
    gt_records = ground_truth[
        ground_truth["frame_id"] == row.frame_id
    ].to_dict("records")

    ground_truth_panel = draw_boxes(
        image=image,
        records=gt_records,
        color=COLORS["ground_truth"],
        show_confidence=False,
    )

    panels.append(
        (
            "Vérité terrain",
            ground_truth_panel,
        )
    )

    # Prédictions des cinq modèles
    for model_name in MODEL_NAMES:
        threshold = selected_thresholds[
            model_name
        ]

        records = all_predictions[
            (
                all_predictions["model"]
                == model_name
            )
            & (
                all_predictions["frame_id"]
                == row.frame_id
            )
            & (
                all_predictions["confidence"]
                >= threshold
            )
        ].to_dict("records")

        prediction_panel = draw_boxes(
            image=image,
            records=records,
            color=COLORS[model_name],
            show_confidence=True,
        )

        panels.append(
            (
                model_name,
                prediction_panel,
            )
        )

    # Une image de vérité terrain + cinq modèles
    number_of_panels = len(panels)

    figure, axes = plt.subplots(
        1,
        number_of_panels,
        figsize=(
            5 * number_of_panels,
            5,
        ),
    )

    axes = np.atleast_1d(axes)

    for axis, (title, panel) in zip(
        axes,
        panels,
    ):
        axis.imshow(panel)
        axis.set_title(title)
        axis.axis("off")

    figure.suptitle(
        (
            f"{row.class_name} — "
            f"{row.frame_filename}"
        ),
        fontsize=13,
    )

    figure.tight_layout()

    output_path = (
        FIGURE_DIR
        / f"qualitative_{row.frame_id}.png"
    )

    figure.savefig(
        output_path,
        dpi=150,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)

    print("Figure sauvegardée :", output_path)


print("Exemples qualitatifs terminés.")

In [ ]:
# ============================================================
# GRAPHIQUE UNIQUE — COMPROMIS QUALITÉ / VITESSE
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. RECHERCHE DU TABLEAU FINAL
# ------------------------------------------------------------

required_columns = {
    "model",
    "f1",
    "map50",
    "latency_mean_ms",
}

summary_candidates = []

for variable_name, variable_value in globals().copy().items():
    if (
        isinstance(variable_value, pd.DataFrame)
        and required_columns.issubset(
            variable_value.columns
        )
    ):
        summary_candidates.append(
            (
                variable_name,
                variable_value.copy(),
            )
        )

if not summary_candidates:
    raise RuntimeError(
        "Le tableau des résultats finaux est introuvable."
    )

summary_name, results_df = max(
    summary_candidates,
    key=lambda candidate: candidate[1]["model"].nunique(),
)

# Si une colonne indique le split, on conserve uniquement le test.
for split_column in [
    "benchmark_split",
    "dataset_split",
    "split",
]:
    if split_column in results_df.columns:
        test_mask = (
            results_df[split_column]
            .astype(str)
            .str.lower()
            == "test"
        )

        if test_mask.any():
            results_df = results_df[
                test_mask
            ].copy()

results_df = (
    results_df
    .drop_duplicates(subset=["model"])
    .copy()
)

# Conversion de la latence moyenne en images par seconde.
results_df["fps"] = (
    1000.0
    / results_df["latency_mean_ms"].astype(float)
)

print("Tableau utilisé :", summary_name)

display(
    results_df[
        [
            "model",
            "f1",
            "map50",
            "latency_mean_ms",
            "fps",
        ]
    ].sort_values(
        "f1",
        ascending=False,
    )
)


# ------------------------------------------------------------
# 2. NOMS D'AFFICHAGE
# ------------------------------------------------------------

DISPLAY_NAMES = {
    "owlv2": "OWLv2",
    "yoloe_26s": "YOLOE-26s",
    "yolo_world": "YOLO-World",
    "omdet_turbo": "OmDet-Turbo",
    "grounding_dino": "Grounding DINO",
}

results_df["display_name"] = (
    results_df["model"]
    .map(DISPLAY_NAMES)
    .fillna(results_df["model"])
)


# ------------------------------------------------------------
# 3. TAILLE DES POINTS SELON LA MAP@0,50
# ------------------------------------------------------------

# La surface des points varie selon la mAP@0,50.
results_df["point_size"] = (
    180
    + 1200
    * results_df["map50"].astype(float)
)


# ------------------------------------------------------------
# 4. CRÉATION DU GRAPHIQUE UNIQUE
# ------------------------------------------------------------

figure, axis = plt.subplots(
    figsize=(11, 7),
)

figure.patch.set_facecolor("white")
axis.set_facecolor("#FAFAFA")

axis.scatter(
    results_df["fps"],
    results_df["f1"],
    s=results_df["point_size"],
    color="#667D89",
    edgecolor="#263A43",
    linewidth=1.3,
    alpha=0.88,
    zorder=3,
)


# ------------------------------------------------------------
# 5. LIBELLÉS DES MODÈLES
# ------------------------------------------------------------

label_offsets = {
    "owlv2": (12, 13),
    "yoloe_26s": (12, 15),
    "yolo_world": (12, -24),
    "omdet_turbo": (12, 13),
    "grounding_dino": (12, -23),
}

for row in results_df.itertuples(index=False):
    x_offset, y_offset = label_offsets.get(
        row.model,
        (10, 10),
    )

    label = (
        f"{row.display_name}\n"
        f"F1 = {row.f1:.3f} | "
        f"mAP = {row.map50:.3f}"
    )

    axis.annotate(
        label,
        xy=(row.fps, row.f1),
        xytext=(x_offset, y_offset),
        textcoords="offset points",
        fontsize=9,
        color="#263238",
        ha="left",
        va="center",
        bbox={
            "boxstyle": "round,pad=0.28",
            "facecolor": "white",
            "edgecolor": "#CFD8DC",
            "alpha": 0.92,
        },
    )


# ------------------------------------------------------------
# 6. MISE EN FORME
# ------------------------------------------------------------

# L'échelle logarithmique est nécessaire, car certains modèles
# traitent environ une image par seconde et d'autres plusieurs
# dizaines d'images par seconde.
axis.set_xscale("log")

axis.set_xlabel(
    "Vitesse d’inférence (images par seconde, échelle logarithmique)",
    fontsize=11,
)

axis.set_ylabel(
    "Score F1 sur le lot de test",
    fontsize=11,
)

axis.set_title(
    "Compromis entre qualité de détection et vitesse d’inférence",
    fontsize=14,
    fontweight="bold",
    pad=15,
)

axis.grid(
    visible=True,
    which="major",
    color="#DCE1E4",
    linewidth=0.8,
    zorder=0,
)

axis.grid(
    visible=False,
    which="minor",
)

axis.set_ylim(
    0,
    max(
        0.45,
        results_df["f1"].max() + 0.08,
    ),
)


# ------------------------------------------------------------
# 7. INDICATION DE LA DIRECTION RECHERCHÉE
# ------------------------------------------------------------

axis.annotate(
    "Compromis recherché :\nplus rapide et plus précis",
    xy=(0.96, 0.94),
    xytext=(0.70, 0.79),
    xycoords="axes fraction",
    textcoords="axes fraction",
    fontsize=9,
    ha="center",
    color="#455A64",
    arrowprops={
        "arrowstyle": "->",
        "color": "#607D8B",
        "linewidth": 1.2,
    },
)


# ------------------------------------------------------------
# 8. LÉGENDE DE LA TAILLE DES POINTS
# ------------------------------------------------------------

legend_map_values = [0.10, 0.30, 0.50]

legend_handles = [
    axis.scatter(
        [],
        [],
        s=180 + 1200 * value,
        color="#667D89",
        edgecolor="#263A43",
        alpha=0.88,
        label=f"{value:.2f}",
    )
    for value in legend_map_values
]

axis.legend(
    handles=legend_handles,
    title="Taille du point : mAP@0,50",
    loc="lower right",
    frameon=True,
    facecolor="white",
    edgecolor="#CFD8DC",
    fontsize=8,
    title_fontsize=9,
)


# ------------------------------------------------------------
# 9. SAUVEGARDE
# ------------------------------------------------------------

figure.tight_layout()

png_output = (
    Path(FIGURE_DIR)
    / "comparaison_qualite_vitesse.png"
)

pdf_output = (
    Path(FIGURE_DIR)
    / "comparaison_qualite_vitesse.pdf"
)

figure.savefig(
    png_output,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

figure.savefig(
    pdf_output,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()
plt.close(figure)

print("PNG :", png_output)
print("PDF :", pdf_output)

In [ ]:
# CELLULE 14 — Tests système et export final
assert len(final_results) == 3
assert set(final_results["model"]) == set(MODEL_NAMES)
assert final_results["selected_threshold"].between(0.01, 0.99).all()
for metric in ["precision", "recall", "f1", "macro_f1", "map50"]:
    assert final_results[metric].between(0, 1).all(), metric
assert (final_results["latency_mean_ms"] > 0).all()
assert (OUTPUT_DIR / "final_test_metrics.csv").exists()
assert (FIGURE_DIR / "final_detection_metrics.png").exists()

protocol = {
    "number_of_frames": 96,
    "calibration_frames": 48,
    "test_frames": 48,
    "number_of_videos": 24,
    "objects_only": True,
    "excluded_class": "child",
    "iou_matching_threshold": MATCH_IOU,
    "nms_iou_threshold": NMS_IOU,
    "raw_confidence_floor": RAW_CONFIDENCE,
    "threshold_selection": (
        "maximum calibration micro-F1; ties resolved by recall, "
        "then precision, then lower threshold"
    ),
    "models": MODEL_NAMES,
    "selected_thresholds": selected_thresholds,
    "best_model_by_rule": str(final_results.iloc[0]["model"]),
}
with open(OUTPUT_DIR / "experimental_protocol.json", "w", encoding="utf-8") as handle:
    json.dump(protocol, handle, indent=2, ensure_ascii=False)

report_lines = [
    "COMPARAISON DES DÉTECTEURS D'OBJETS — RÉSULTATS TEST",
    "=" * 60,
    final_results[
        [
            "model", "selected_threshold", "precision", "recall",
            "f1", "macro_f1", "map50", "fp_per_frame",
            "latency_mean_ms", "latency_p95_ms", "fps_from_mean"
        ]
    ].round(4).to_string(index=False),
    "",
    f"Modèle retenu par la règle annoncée : {final_results.iloc[0]['model']}",
    "",
    "Important : les seuils ont été choisis sur la calibration ; "
    "les valeurs ci-dessus proviennent uniquement du test.",
]
(OUTPUT_DIR / "report_ready_summary.txt").write_text(
    "\n".join(report_lines), encoding="utf-8"
)

archive_path = Path("/kaggle/working/detector_benchmark_comparison_results.zip")
if archive_path.exists():
    archive_path.unlink()
shutil.make_archive(
    str(archive_path.with_suffix("")),
    "zip",
    root_dir=OUTPUT_DIR,
)

print("Tests système : réussis")
print("Archive à télécharger :", archive_path)
print("Envoie-moi ensuite :")
print("  1. detector_benchmark_comparison_results.zip")
print("  2. une capture du tableau final affiché par la cellule 11")

## Interprétation à respecter

Le modèle retenu n'est pas automatiquement celui ayant la meilleure
précision isolée. Pour un système de sécurité, un faux négatif peut être
plus grave qu'un faux positif ; la décision finale doit donc considérer
conjointement le **rappel**, le **F1**, les faux positifs par image et la
latence. Le classement automatique du notebook utilise d'abord le F1,
puis le rappel, puis la latence. Ce choix devra être discuté dans le
rapport à partir des résultats réellement obtenus.

Sources techniques :

- [Grounding DINO — documentation Transformers](https://huggingface.co/docs/transformers/model_doc/grounding-dino)
- [YOLO-World — documentation Ultralytics](https://docs.ultralytics.com/models/yolo-world/)
- [OWLv2 — documentation Transformers](https://huggingface.co/docs/transformers/model_doc/owlv2)